# 02 — Sentence Transformer positive-pair baseline

Consumes the canonical CSV written by Notebook 01. This notebook encodes a reproducible sample of explicitly relevant synthetic pairs and reports descriptive cosine-similarity values only. These values are not accuracy, probabilities, or hiring-performance metrics because the dataset contains no verified negative pairs.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [start, *start.parents] if (p / "manage.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside AI_RESUME_ANALYSIS or one of its subfolders.")
sys.path.insert(0, str(PROJECT_ROOT))
pairs_path = PROJECT_ROOT / "data" / "processed" / "resume_job_positive_pairs.csv"
if not pairs_path.is_file():
    raise FileNotFoundError("Run notebooks/01_EDA_and_DATA_prep.ipynb first.")
pairs = pd.read_csv(pairs_path)
assert set(pairs["match_label"]) == {"explicit_relevant"}
print(f"Loaded {len(pairs):,} explicitly relevant pairs from Notebook 01.")


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_PATH = PROJECT_ROOT / "ml" / "models" / "bert_resume_matcher"
model = SentenceTransformer(str(MODEL_PATH) if MODEL_PATH.is_dir() else MODEL_ID, device="cpu")
sample = pairs.sample(n=min(500, len(pairs)), random_state=42).reset_index(drop=True)
resume_embeddings = model.encode(sample["resume_text"].tolist(), batch_size=32, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
job_embeddings = model.encode(sample["job_description"].tolist(), batch_size=32, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
sample["cosine_similarity"] = np.sum(resume_embeddings * job_embeddings, axis=1)
summary = sample["cosine_similarity"].describe(percentiles=[.25, .5, .75]).round(4)
print("Model:", MODEL_ID)
print("Pair sample size:", len(sample))
print("Descriptive similarity among explicit positives only:")
print(summary)
print("This is not a held-out quality metric; no negative labels are available.")
out = PROJECT_ROOT / "data" / "processed" / "positive_similarity_preview.csv"
sample[["resume_id", "job_id", "match_label", "cosine_similarity"]].to_csv(out, index=False)
print("Saved:", out)
